# Lesson 2c — `nn.Embedding` from scratch (runnable)

The first "no black boxes" lesson. We re-implement `nn.Embedding` from scratch in ~15 lines and prove it gives the same forward result, same gradients, and trains to the same answer as PyTorch's built-in.

Runnable version of [`02c_embedding_from_scratch.py`](../02c_embedding_from_scratch.py).


## Imports

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False)

## Step 1 — `nn.Embedding` as PyTorch gives it to us

In [ ]:
V, d = 5, 3
ref = nn.Embedding(V, d)
print(f"nn.Embedding({V}, {d}) — V words, d-dim each")
print()
print("What's inside?")
for name, p in ref.named_parameters():
    print(f"  parameter '{name}' has shape {tuple(p.shape)}:")
    print(p.data)
print()

ids = torch.tensor([0, 2, 4])
vectors = ref(ids)
print(f"ref({ids.tolist()}) returns:")
print(vectors)
print(f"shape: {tuple(vectors.shape)}")

## Step 2 — Prove it's just indexing

`nn.Embedding` is a SINGLE WEIGHT MATRIX. When you call `emb(ids)`, it returns `weight[ids]`. That's it. No math. Just memory lookup.

In [ ]:
print("These should be identical:")
print(f"  ref(ids):         {ref(ids).flatten().tolist()}")
print(f"  ref.weight[ids]:  {ref.weight[ids].flatten().tolist()}")
print()
print("That's the whole magic — just indexing.")

## Step 3 — Re-implement `nn.Embedding` ourselves

An `nn.Module` is two things: parameters + a forward method.

In [ ]:
class MyEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        # The ONE parameter: a (vocab_size, dim) trainable matrix.
        self.weight = nn.Parameter(torch.randn(vocab_size, dim))

    def forward(self, ids):
        return self.weight[ids]             # The forward pass IS the lookup.


# Copy ref's weights so we can compare exactly
mine = MyEmbedding(V, d)
mine.weight.data.copy_(ref.weight.data)

print("Same input, same weights, same output:")
print(f"  ref(ids):   {ref(ids).flatten().tolist()}")
print(f"  mine(ids):  {mine(ids).flatten().tolist()}")
assert torch.allclose(ref(ids), mine(ids))
print("\n  ✓ Outputs identical.")

## Step 4 — Gradients

Only the ROWS we LOOKED UP get a gradient. Repeated lookups ACCUMULATE.

In [ ]:
mine2 = MyEmbedding(V, d)
ref2  = nn.Embedding(V, d)
ref2.weight.data.copy_(mine2.weight.data)

ids = torch.tensor([0, 2, 2, 4])            # id 2 appears TWICE.
out_mine = mine2(ids).sum()
out_ref  = ref2(ids).sum()
out_mine.backward()
out_ref.backward()

print("Lookup ids:", ids.tolist())
print("Reference (nn.Embedding) gradient:")
print(ref2.weight.grad)
print("\nOur version gradient:")
print(mine2.weight.grad)
print()
print("Row 2 has gradient [2, 2, 2] — looked up TWICE, summed.")
print("Rows 1 and 3 have gradient 0 — never looked up.")
assert torch.allclose(ref2.weight.grad, mine2.weight.grad)
print("✓ Gradients identical.")

## Step 5 — Train both on the same task

Make sure they converge to the same answer.

In [ ]:
def make_dataset(V, n=200):
    ids = torch.randint(0, V, (n,))
    labels = (ids % 2).long()                # Trivial: predict parity.
    return ids, labels

def train_one(emb_module, V, d=8, n_classes=2, steps=300):
    torch.manual_seed(123)
    ids, y = make_dataset(V, n=200)
    head = nn.Linear(d, n_classes)
    opt = torch.optim.AdamW(list(emb_module.parameters()) + list(head.parameters()), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(steps):
        h = emb_module(ids)
        logits = head(h)
        loss = loss_fn(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item(), (head(emb_module(ids)).argmax(-1) == y).float().mean().item()

V_big, d_big = 20, 8
torch.manual_seed(42); ref_emb = nn.Embedding(V_big, d_big)
torch.manual_seed(42); mine_emb = MyEmbedding(V_big, d_big)

loss_ref,  acc_ref  = train_one(ref_emb,  V_big, d_big)
loss_mine, acc_mine = train_one(mine_emb, V_big, d_big)

print(f"nn.Embedding (PyTorch): loss {loss_ref:.4f}  accuracy {acc_ref*100:.1f}%")
print(f"MyEmbedding (ours):     loss {loss_mine:.4f}  accuracy {acc_mine*100:.1f}%")
print()
print("Our 4-line class does everything nn.Embedding does. Same math.")